In [10]:
import pandas as pd
import csv
from pathlib import Path

In [11]:
DATA_DIR = Path("./../../data/AusPAR")
FILE_GLOB = "*.csv"
SAVE_MERGED = True
MERGED_OUT = DATA_DIR / "AusPAR_merged.csv"

In [12]:
def into_df(input):
    return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True, on_bad_lines='skip')

In [13]:
# inspect an example file

input_path = "./../../data/AusPAR/COGNOS_V_GEN_PRODUCT.csv"
df = into_df(input_path)
print(len(df))
df.columns

31942


Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE',
       'PRODUCT_CEASED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO',
       'ADDITIONAL_WARNING_INFO', 'PRODUCT_CODE', 'CREATION_DATE',
       'LAST_UPDATE_DATE'],
      dtype='object')

In [14]:

csv_paths = sorted(DATA_DIR.glob(FILE_GLOB))
if not csv_paths:
    raise FileNotFoundError(f"No files matching {FILE_GLOB} in {DATA_DIR}")

dfs = []
schema_summary = {}
skipped_files  = []

for p in csv_paths:
    df = into_df(p)
    if "PRODUCT_ID" not in df.columns:
        skipped_files.append(p.name)
        continue

    dfs.append(df)
    schema_summary[p.name] = set(df.columns)

if not dfs:
    raise RuntimeError("No CSV contained a 'PRODUCT_ID' column — nothing to merge.")

print(f"will merge {len(dfs)} file(s) that contain PRODUCT_ID")
if skipped_files:
    print("skipped (no PRODUCT_ID):", ", ".join(skipped_files))


all_cols   = set().union(*schema_summary.values())
common_cols = set.intersection(*schema_summary.values())
print("──────────────────")
print(f"│  files found     : {len(csv_paths)}")
print(f"│  union of columns: {len(all_cols)}")
print(f"│  common to all   : {len(common_cols)}, {sorted(common_cols)}")
print("──────────────────")


normed = []
for df in dfs:
    # add any absent columns as NaN so every df has identical schema
    for col in all_cols - set(df.columns):
        df[col] = pd.NA
    normed.append(df[sorted(all_cols)])

merged_df = pd.concat(normed, ignore_index=True)

merged_df.to_csv(
    MERGED_OUT,
    index=False,
    sep=",",                   # standard CSV
    quoting=csv.QUOTE_MINIMAL  # only quote when needed
)
print(f"written to: {MERGED_OUT.resolve()}")


/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_64356/1824610912.py:2: DtypeWarning: Columns (19,20,24,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True, on_bad_lines='skip')


will merge 3 file(s) that contain PRODUCT_ID
skipped (no PRODUCT_ID): AusPAR_merged.csv, COGNOS_V_GEN_COMPONENT_ADMIN_ROUTE.csv, COGNOS_V_GEN_FORMULATION.csv, COGNOS_V_GEN_INGREDIENT.csv, COGNOS_V_GEN_LICENCE.csv
──────────────────
│  files found     : 8
│  union of columns: 25
│  common to all   : 3, ['CREATION_DATE', 'LAST_UPDATE_DATE', 'PRODUCT_ID']
──────────────────


/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_64356/2939363013.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat(normed, ignore_index=True)


written to: /Users/shtosti/Dropbox/Projects/DrugFork/data/AusPAR/AusPAR_merged.csv


In [ ]:
merged_df = pd.read_csv(MERGED_OUT, delimiter=',', encoding='utf-8')
cols_to_drop = ['LAST_UPDATE_DATE', 
                'SHELF_LIFE_CONTAINER_INFO', 
                'WEIGHT_OF_DIVIDED_PREPARATION', 
                'MAX_DAILY_DOSE', 
                'MAX_DAILY_DOSE_UNIT', 
                'MAX_SINGLE_DOSE', 
                'MAX_SINGLE_DOSE_UNIT', 
                'PRODUCT_SUPPLIED_DATE',
                'DOSAGE_FORM_CODE'
                ]
merged_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
merged_df.columns

/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_64356/2294610750.py:1: DtypeWarning: Columns (0,2,6,7,11,13,15,17,18,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv(MERGED_OUT, delimiter=',', encoding='utf-8')


Index(['ADDITIONAL_INFO', 'ADDITIONAL_WARNING_INFO', 'COMPONENT_DESCRIPTION',
       'COMPONENT_ID', 'COMPONENT_NUMBER', 'CREATION_DATE', 'DOSAGE_FORM_CODE',
       'INDICATION_TEXT', 'LICENCE_ID', 'PRODUCT_CEASED_DATE', 'PRODUCT_CODE',
       'PRODUCT_ID', 'PRODUCT_NAME', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_TYPE',
       'VISUAL_IDENTIFICATION'],
      dtype='object')

In [16]:
licence_df = pd.read_csv(
    "./../../data/AusPAR/COGNOS_V_GEN_LICENCE.csv",
    delimiter='~',
    encoding='utf-8',
    skipinitialspace=True
)

component_route_df = pd.read_csv(
    "./../../data/AusPAR/COGNOS_V_GEN_COMPONENT_ADMIN_ROUTE.csv",
    delimiter='~',
    encoding='utf-8',
    skipinitialspace=True
)

/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_64356/2522812802.py:1: DtypeWarning: Columns (19,20,24,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  licence_df = pd.read_csv(


In [17]:
# merge by LICENCE_ID
merged_df = pd.merge(
    merged_df,
    licence_df[['SPONSOR_ID', 'LICENCE_ID', 'LICENCE_STATUS']],
    on='LICENCE_ID',
    how='left'
)

merged_df.columns

Index(['ADDITIONAL_INFO', 'ADDITIONAL_WARNING_INFO', 'COMPONENT_DESCRIPTION',
       'COMPONENT_ID', 'COMPONENT_NUMBER', 'CREATION_DATE', 'DOSAGE_FORM_CODE',
       'INDICATION_TEXT', 'LICENCE_ID', 'PRODUCT_CEASED_DATE', 'PRODUCT_CODE',
       'PRODUCT_ID', 'PRODUCT_NAME', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_TYPE',
       'VISUAL_IDENTIFICATION', 'SPONSOR_ID', 'LICENCE_STATUS'],
      dtype='object')

In [18]:
# merge by ROUTE_OF_ADMIN_CODE
merged_df = pd.merge(
    merged_df,
    component_route_df[['ROUTE_OF_ADMIN_CODE', 'COMPONENT_ID']],
    on='COMPONENT_ID',
    how='left'
)

merged_df.columns

Index(['ADDITIONAL_INFO', 'ADDITIONAL_WARNING_INFO', 'COMPONENT_DESCRIPTION',
       'COMPONENT_ID', 'COMPONENT_NUMBER', 'CREATION_DATE', 'DOSAGE_FORM_CODE',
       'INDICATION_TEXT', 'LICENCE_ID', 'PRODUCT_CEASED_DATE', 'PRODUCT_CODE',
       'PRODUCT_ID', 'PRODUCT_NAME', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_TYPE',
       'VISUAL_IDENTIFICATION', 'SPONSOR_ID', 'LICENCE_STATUS',
       'ROUTE_OF_ADMIN_CODE'],
      dtype='object')

In [ ]:
merged_df.to_csv(
    MERGED_OUT,
    index=False,
    sep=",",
    quoting=csv.QUOTE_MINIMAL
)
print(f"written to: {MERGED_OUT.resolve()}")